# Diffusion — decay of a magnetic Fourier mode — 3D

`MagneticStandalone3D` · ASN 28 validation case (TRUST-LiMe).

## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``diffusion``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone3D` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

$b(x,y,0)=\sin(\pi y)$, Dirichlet in $Y$, zero flux in $X$, $u=0$, $\eta=0.05$:

$$b(x,y,t)=\sin(\pi y)\,e^{-\eta\pi^2t}.$$

We compare the *amplitude ratio* $b(t)/b(0)$ (independent of where the probe is
snapped) to the analytic factor, over all recorded times, and measure the
relaxation rate to make sure the transport coefficient is $\eta$ itself.

Mesh convergence (against `diffusion_refined`) is checked in the
`diffusion_refined` notebook.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 1 | transport coefficient is $\eta$ itself — **no** $\rho C_p$ leak from the energy base class | measured rate vs $\eta\pi^2$ |
| 2 | $\eta$ used verbatim as diffusion coefficient | decay rate |
| 8 | Dirichlet prescribes $b$; Neumann = zero flux | implied analytic rate |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "diffusion.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition diffusion.data 2 -> PAR_diffusion.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
ETA, K = 0.05, np.pi
def analytic_ratio(t): return np.exp(-ETA * K**2 * np.asarray(t, float))

TAGS = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
err_coarse = ratio_error(case.dataFileName_, TAGS, analytic_ratio)
print(f"coarse  max |ratio - analytic| : {err_coarse:.4e}")
assert err_coarse < 0.05, f"coarse decay error too large: {err_coarse:.3e}"
print("OK: decay matches the analytic solution.")

# --- Business contract: the transport coefficient is eta ITSELF -------
# The class derives from the ENERGY equation, so the real danger is an
# inherited rho*Cp time factor (~1.2e3 for air). Measure the actual
# relaxation rate lambda = -d/dt ln(b/b0) and compare with eta*pi^2.
t, b = read_son(case.dataFileName_, "SONDE_Y050")
lam_measured = -np.log(abs(b[-1] / b[0])) / t[-1]
lam_expected = ETA * np.pi**2
rel = abs(lam_measured - lam_expected) / lam_expected
print(f"measured relaxation rate : {lam_measured:.6f} (expected {lam_expected:.6f})")
print(f"relative error           : {rel:.3e}")
assert rel < 0.05, f"transport coefficient is not eta: rel. error {rel:.3e}"
print("OK: transport coefficient is exactly eta (no rho*Cp leak).")

import matplotlib.pyplot as plt
t, b = read_son(case.dataFileName_, "SONDE_Y050")
plt.figure(figsize=(6,4))
plt.plot(t, b/b[0], "o-", ms=2, label="TRUST-LiMe (coarse)")
plt.plot(t, analytic_ratio(t), "k--", label="analytic exp(-eta pi^2 t)")
plt.xlabel("t [s]"); plt.ylabel("b(t)/b(0)"); plt.legend(); plt.grid(True); plt.show()


## Computer Performance


In [ ]:
run.tablePerf()